# 04 — Diagnosa: Kenapa Sinyal RSI14+RSI7 Rugi di 2025–2026?

Notebook 03: top-5 risk-on ekspektasi **+4,61%** (2024) lalu **−1,62%** (2025) dan **−1,75%** (2026 s/d Juni).
Sebelum memperbaiki apa pun, cari tahu **sumber** kerugiannya.

### Hipotesis
| # | Hipotesis | Cara uji |
|---|---|---|
| H1 | **Beta pasar**: coin pilihan tetap lebih baik dari rata-rata coin lain, tapi seluruh pasar alt turun | pecah return sinyal = return rata-rata semua coin (benchmark) + **alpha** (selisih) di jendela waktu yang sama |
| H2 | **Momentum berbalik**: coin RSI tertinggi justru kalah dari coin lain (mean reversion) | spread return 7 hari peringkat 1–5 vs 35–39 per tahun |
| H3 | **Coin baru listing** mendominasi top dan dump | ekspektasi per umur listing per tahun |
| H4 | **Exit** tidak cocok rezim baru | return di horizon tetap 1/3/7/14 hari per tahun |

### Kandidat perbaikan (ditetapkan sebelum melihat hasil; dipilih di dev 2021–2023)
| Filter (ditambahkan ke H8b) | Definisi (data s/d hari sinyal) |
|---|---|
| F1 breadth | ≥ 50% coin universe close > SMA50 |
| F2 musim alt | median return 30 hari alt > return 30 hari BTC |
| F3 tren BTC pendek | close BTC > SMA50 |

Diterima kalau ekspektasi dev **naik** dan sisa sinyal ≥ 50%. ⚠️ **2025–2026 sudah "terlihat"** (dipakai untuk
menemukan masalah ini), jadi hasil di periode itu bukan lagi out-of-sample murni — patokan jujur: dev + 2024 tetap
tidak boleh rusak, dan perbaikan wajib dibuktikan lewat paper trading.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from data_source.fear_greed import load_fear_greed_index

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
STABLE = {"BFUSDUSDT", "RLUSDUSDT", "USD1USDT", "USDCUSDT", "USDEUSDT", "USDSUSDT", "UUSDT"}
GOLD, DUP = {"PAXGUSDT", "XAUTUSDT"}, {"WBETHUSDT"}
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))
UNIV = [s for s in ALL_SYMS if s not in STABLE | GOLD | DUP]
BTC, COST = "BTCUSDT", 0.0015
COINS_NB = [c for c in UNIV if c != BTC]

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in UNIV}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
close_df = pd.DataFrame({s: raw[s]["close"].reindex(dates) for s in UNIV})
open_df = pd.DataFrame({s: raw[s]["open"].reindex(dates) for s in UNIV})
low_df = pd.DataFrame({s: raw[s]["low"].reindex(dates) for s in UNIV})

fgi = load_fear_greed_index()["fng_value"].reindex(dates).ffill()
btc = close_df[BTC]
market_ok = (btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)

def rsi(s, n=14):
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

score_df = (close_df.apply(rsi, n=14).rank(axis=1, pct=True) + close_df.apply(rsi, n=7).rank(axis=1, pct=True)) / 2

# kandidat filter (semua memakai data s/d hari sinyal)
above50 = (close_df > close_df.rolling(50, min_periods=50).mean()).where(close_df.rolling(50, min_periods=50).mean().notna())
breadth = above50[COINS_NB].mean(axis=1)
alt_ret30 = (close_df[COINS_NB] / close_df[COINS_NB].shift(30) - 1).median(axis=1)
btc_ret30 = btc / btc.shift(30) - 1
FILT = {
    "F1 breadth >= 50%": breadth >= 0.5,
    "F2 musim alt": alt_ret30 > btc_ret30,
    "F3 BTC > SMA50": btc > btc.rolling(50, min_periods=50).mean(),
}
first_valid = close_df.apply(lambda s: s.first_valid_index())
print(f"Universe {len(UNIV)} coin, kandidat {len(COINS_NB)}")

Universe 40 coin, kandidat 39


## Mesin sinyal + benchmark

Sama persis dengan notebook 02/03 (top-5, exit skor < 0,30 / 14 hari / stop −25%), ditambah untuk tiap sinyal:
**benchmark** = rata-rata return semua coin kandidat lain yang punya harga, pada jendela masuk–keluar yang sama
(harga keluar memakai jenis harga yang sama: open kalau exit skor/stop, close kalau maks hari), dan return BTC.
**Alpha** = return sinyal − benchmark. Juga dihitung return horizon tetap 1/3/7/14 hari dari open masuk.
Untuk H2 dihitung juga sinyal peringkat **35–39 (paling bawah)** dengan aturan exit yang sama.

In [2]:
CLOSE, OPEN, LOW = (df[UNIV].to_numpy(float) for df in (close_df, open_df, low_df))
SCORE = score_df[UNIV].to_numpy(float)
NB_IDX = np.array([UNIV.index(c) for c in COINS_NB])
BTC_I = UNIV.index(BTC)
FLOOR, MAX_HOLD, HARD_STOP, K = 0.30, 14, 0.25, 5
T = len(dates)

def bench(i, exit_i, use_open):
    px_out = OPEN[exit_i, NB_IDX] if use_open else CLOSE[exit_i, NB_IDX]
    r = px_out / OPEN[i + 1, NB_IDX] - 1
    return np.nanmean(r)

rows = []
for i in range(T - 2):
    row = SCORE[i, NB_IDX]
    valid = ~np.isnan(row)
    if valid.sum() < 2 * K:
        continue
    order = np.argsort(np.where(valid, row, -np.inf))[::-1]
    top = [(r + 1, jj) for r, jj in enumerate(order[:K])]
    nv = int(valid.sum())
    bottom = [(nv - k, order[nv - 1 - k]) for k in range(K)]      # peringkat paling bawah (yang valid)
    for grp, items in (("top", top), ("bottom", bottom)):
        for rank, jj in items:
            ci = NB_IDX[jj]
            entry = OPEN[i + 1, ci]
            if np.isnan(entry) or entry <= 0:
                continue
            stop = entry * (1 - HARD_STOP)
            exit_px, reason, exit_i, use_open = None, None, None, True
            end = min(i + 1 + MAX_HOLD, T)
            for day in range(i + 1, end):
                lo, op = LOW[day, ci], OPEN[day, ci]
                if not np.isnan(lo) and lo <= stop:
                    exit_px = op if (day != i + 1 and not np.isnan(op) and op <= stop) else stop
                    reason, exit_i = "STOP_KERAS", day; break
                sc_y = SCORE[day - 1, ci]
                if day > i + 1 and not np.isnan(sc_y) and sc_y < FLOOR:
                    exit_px, reason, exit_i = (op if not np.isnan(op) else CLOSE[day, ci]), "SKOR_TURUN", day; break
            if exit_px is None:
                exit_i = end - 1
                exit_px = CLOSE[exit_i, ci]
                if np.isnan(exit_px):
                    continue
                reason, use_open = "MAKS_HARI", False
            ret = exit_px * (1 - COST) / (entry * (1 + COST)) - 1
            hz = {}
            for h in (1, 3, 7, 14):
                j = i + h
                hz[f"r{h}"] = CLOSE[j, ci] / entry - 1 if j < T else np.nan
            b_px = OPEN[exit_i, BTC_I] if use_open else CLOSE[exit_i, BTC_I]
            rows.append(dict(i=i, date=dates[i], grp=grp, rank=rank, coin=UNIV[ci], market_ok=bool(market_ok.iloc[i]),
                             age=(dates[i] - first_valid[UNIV[ci]]).days, reason=reason, ret=ret,
                             bench=bench(i, exit_i, use_open) - 2 * COST, btc=b_px / OPEN[i + 1, BTC_I] - 1, **hz))
sig = pd.DataFrame(rows)
sig["alpha"] = sig.ret - sig.bench
sig["year"] = sig.date.dt.year
top_on = sig[(sig.grp == "top") & sig.market_ok & (sig.date < "2026-06-30")]
print(f"{len(sig):,} sinyal; top-5 risk-on: {len(top_on):,}")
chk = top_on[(top_on.date >= "2024-01-01")]
print(f"Cek reproduksi uji 2024-26: n={len(chk)} win_rate={(chk.ret > 0).mean():.1%} ekspektasi={chk.ret.mean():+.2%}  (nb02: 2855 / 45.5% / +1.79%)")

19,910 sinyal; top-5 risk-on: 5,640
Cek reproduksi uji 2024-26: n=2855 win_rate=45.5% ekspektasi=+1.79%  (nb02: 2855 / 45.5% / +1.79%)


## H1 — Beta pasar vs alpha pilihan coin (top-5, risk-on)

In [3]:
def pct(x): return f"{x:+.2%}"
g = top_on.groupby("year")
t1 = pd.DataFrame({
    "n": g.size(),
    "win_rate": g.ret.apply(lambda r: f"{(r > 0).mean():.1%}"),
    "return_sinyal": g.ret.mean().map(pct),
    "benchmark_semua_alt": g.bench.mean().map(pct),
    "BTC_jendela_sama": g.btc.mean().map(pct),
    "ALPHA": g.alpha.mean().map(pct),
    "alpha_menang%": g.alpha.apply(lambda a: f"{(a > 0).mean():.1%}"),
})
t1

,n,win_rate,return_sinyal,benchmark_semua_alt,BTC_jendela_sama,ALPHA,alpha_menang%
year,,,,,,,
2021,1220,52.2%,+20.26%,+13.91%,+3.09%,+6.35%,37.3%
2022,130,14.6%,-11.11%,-6.23%,-5.87%,-4.87%,28.5%
2023,1435,44.3%,+0.99%,+1.63%,+2.13%,-0.65%,39.7%
2024,1565,47.6%,+4.61%,+3.93%,+2.96%,+0.68%,40.6%
2025,1065,44.0%,-1.62%,-1.08%,+0.73%,-0.54%,44.4%
2026,225,37.3%,-1.75%,-2.02%,-4.90%,+0.27%,37.8%


## H2 — Apakah momentum berbalik? Top-5 vs peringkat paling bawah (risk-on, exit sama)

In [4]:
both = sig[sig.market_ok & (sig.date < "2026-06-30")]
p = both.pivot_table(index="year", columns="grp", values=["ret", "r7"], aggfunc="mean")
t2 = pd.DataFrame({
    "top5_exit_skor": p[("ret", "top")].map(pct), "bawah5_exit_skor": p[("ret", "bottom")].map(pct),
    "spread_exit": (p[("ret", "top")] - p[("ret", "bottom")]).map(pct),
    "top5_7hari": p[("r7", "top")].map(pct), "bawah5_7hari": p[("r7", "bottom")].map(pct),
    "spread_7hari": (p[("r7", "top")] - p[("r7", "bottom")]).map(pct),
})
t2

,top5_exit_skor,bawah5_exit_skor,spread_exit,top5_7hari,bawah5_7hari,spread_7hari
year,,,,,,
2021,+20.26%,+0.45%,+19.82%,+11.39%,+3.60%,+7.78%
2022,-11.11%,-1.49%,-9.62%,-7.26%,-3.82%,-3.43%
2023,+0.99%,-0.22%,+1.20%,+0.88%,+0.62%,+0.26%
2024,+4.61%,-0.34%,+4.95%,+3.71%,+0.84%,+2.87%
2025,-1.62%,-0.89%,-0.73%,-0.07%,-1.25%,+1.17%
2026,-1.75%,-0.33%,-1.42%,-0.17%,-1.22%,+1.05%


## H3 — Umur listing (top-5, risk-on)

In [5]:
top_on = top_on.assign(umur=pd.cut(top_on.age, [-1, 180, 365, 10_000], labels=["< 6 bln", "6-12 bln", "> 1 thn"]))
t3 = top_on.pivot_table(index="year", columns="umur", values="ret", aggfunc=["count", "mean"])
t3["mean"] = t3["mean"].map(lambda v: pct(v) if pd.notna(v) else "-")
t3

count                      mean                  
umur < 6 bln 6-12 bln > 1 thn  < 6 bln 6-12 bln  > 1 thn
year                                                    
2021   675.0    545.0     NaN  +33.92%   +3.34%        -
2022     NaN      4.0   126.0        -   +1.56%  -11.51%
2023    43.0     13.0  1379.0  -10.27%  +12.46%   +1.23%
2024   105.0    165.0  1295.0   -6.35%  +12.10%   +4.55%
2025    17.0     53.0   995.0  -17.06%   -9.37%   -0.94%
2026     NaN     23.0   202.0        -   -4.42%   -1.45%

## H4 — Return di horizon tetap vs exit skor (top-5, risk-on)

In [6]:
t4 = top_on.groupby("year")[["r1", "r3", "r7", "r14", "ret"]].mean().rename(columns={"ret": "exit_skor"})
ex = top_on.groupby(["year", "reason"]).size().unstack(fill_value=0)
ex = ex.div(ex.sum(axis=1), axis=0).map(lambda v: f"{v:.0%}")
pd.concat([t4.map(pct), ex], axis=1)

,r1,r3,r7,r14,exit_skor,MAKS_HARI,SKOR_TURUN,STOP_KERAS
year,,,,,,,,
2021,+1.65%,+4.64%,+11.39%,+23.08%,+20.26%,54%,25%,21%
2022,-1.18%,-3.52%,-7.26%,-12.52%,-11.11%,52%,25%,23%
2023,+0.33%,+0.76%,+0.88%,+1.98%,+0.99%,66%,28%,6%
2024,+0.60%,+1.69%,+3.71%,+5.73%,+4.61%,61%,22%,17%
2025,+0.16%,+0.26%,-0.07%,-0.30%,-1.62%,68%,18%,14%
2026,+0.77%,+0.48%,-0.17%,-0.69%,-1.75%,61%,26%,13%


## Kandidat perbaikan F1–F3

Pilih di dev 2021–2023 (ekspektasi naik + sisa sinyal ≥ 50%), lalu lihat 2024 dan 2025–26.

In [7]:
PERIODS = {"dev 2021-23": ("2021-01-01", "2023-12-31"), "2024": ("2024-01-01", "2024-12-31"),
           "2025-26 (sudah terlihat)": ("2025-01-01", "2026-06-29")}
def stat(d):
    return dict(n=len(d), win_rate=f"{(d.ret > 0).mean():.1%}" if len(d) >= 20 else "-",
                ekspektasi=pct(d.ret.mean()) if len(d) >= 20 else "-")
rows, accepted = [], []
for name, f in [("H8b saja", None), *FILT.items()]:
    d = top_on if f is None else top_on[f.reindex(top_on.date).to_numpy(bool)]
    for pname, (a, b) in PERIODS.items():
        dd = d[(d.date >= a) & (d.date <= b)]
        base = top_on[(top_on.date >= a) & (top_on.date <= b)]
        rows.append(dict(filter=name, periode=pname, **stat(dd), sisa=f"{len(dd) / len(base):.0%}"))
    if f is not None:
        dv, bv = d[d.date <= "2023-12-31"], top_on[top_on.date <= "2023-12-31"]
        if dv.ret.mean() > bv.ret.mean() and len(dv) / len(bv) >= 0.5:
            accepted.append(name)
print("Lolos aturan dev:", accepted or "TIDAK ADA")
pd.DataFrame(rows).set_index(["filter", "periode"])

Lolos aturan dev: TIDAK ADA


n win_rate ekspektasi  sisa
filter            periode                                                 
H8b saja          dev 2021-23               2785    46.4%     +8.87%  100%
                  2024                      1565    47.6%     +4.61%  100%
                  2025-26 (sudah terlihat)  1290    42.9%     -1.64%  100%
F1 breadth >= 50% dev 2021-23               1845    46.9%     +6.02%   66%
                  2024                       870    44.7%     +5.99%   56%
                  2025-26 (sudah terlihat)   635    46.6%     +0.26%   49%
F2 musim alt      dev 2021-23               1145    39.5%     +7.92%   41%
                  2024                       350    43.1%     -1.83%   22%
                  2025-26 (sudah terlihat)   370    41.1%     -1.17%   29%
F3 BTC > SMA50    dev 2021-23               2035    48.7%     +6.18%   73%
                  2024                      1185    45.4%     +5.19%   76%
                  2025-26 (sudah terlihat)  1010    43.6%     -1.69%   78%

## Kesimpulan

Reproduksi cocok (uji 2024–26: 2855 sinyal, 45,5%, +1,79%).

**H1 — Penyebab utama: beta pasar alt, bukan kemampuan memilih coin.**
| Tahun | Return sinyal | Rata-rata semua alt (jendela sama) | BTC | **Alpha** |
|---|---|---|---|---|
| 2021 | +20,26% | +13,91% | +3,09% | **+6,35%** |
| 2022 | −11,11% | −6,23% | −5,87% | −4,87% |
| 2023 | +0,99% | +1,63% | +2,13% | −0,65% |
| 2024 | +4,61% | +3,93% | +2,96% | +0,68% |
| 2025 | −1,62% | −1,08% | +0,73% | −0,54% |
| 2026 | −1,75% | −2,02% | −4,90% | +0,27% |

Sejak 2022, alpha (keunggulan coin pilihan dibanding membeli rata-rata semua alt) **≈ 0**; untung 2024 sebagian
besar (+3,93 dari +4,61%) datang dari pasar alt yang naik. Rugi 2025 terjadi karena alt turun padahal BTC naik —
filter H8b (berbasis BTC & FGI) tetap "risk-on" sementara alt melemah. Alpha positif hanya di 2021 (reli alt besar).
Median alpha selalu negatif (alpha_menang 28–44%): rata-rata ditopang segelintir coin yang naik sangat besar.

**H2 — Ranking 7 hari belum berbalik, tapi tipis.** Spread top-5 vs bawah-5 di horizon 7 hari 2025 +1,17%,
2026 +1,05% (2024 +2,87%). Dengan aturan exit, spread 2025–26 jadi negatif → sebagian kerugian datang dari exit.

**H3 — Coin baru listing (< 6 bulan) buruk sejak 2023** (2023 −10,3%, 2024 −6,4%, 2025 −17,1%), hanya bagus di
2021. Jumlahnya kecil di 2025 (17 sinyal), dan coin > 1 tahun pun −0,94% di 2025 → bukan penyebab utama.
Filter umur listing tetap ditolak oleh aturan dev (notebook 03), dicatat sebagai pola yang perlu dipantau.

**H4 — Exit skor menggerus hasil di hampir semua tahun.** Hold tetap 14 hari (dikurangi biaya 0,3%) vs exit skor:
2021 +22,8 vs +20,3%, 2023 +1,7 vs +1,0%, 2024 +5,4 vs +4,6%, 2025 −0,6 vs −1,6%; hanya 2022 (bear) exit skor
lebih baik (−11,1 vs −12,8%). Ini temuan baru yang **belum diuji dengan benar** (tanpa stop keras ada risiko ekor;
jumlah percobaan bertambah) → kandidat notebook 05, dipilih di dev.

**Kandidat perbaikan F1–F3: tidak ada yang lolos aturan dev.** F1 (breadth ≥ 50% coin di atas SMA50) menarik —
2024 +5,99%, 2025–26 **+0,26%** (vs −1,64%) — tapi menurunkan dev (+8,87 → +6,02%) dan 2025–26 sudah terlihat,
jadi **tidak diadopsi**; boleh dipantau lewat paper trading. F2 (musim alt) dan F3 (BTC > SMA50) tidak membantu.

**Jumlah percobaan riset rekomendasi**: notebook 03 (8) + notebook 04 (3 filter) = 11 varian, 0 diadopsi.

### Arti untuk fitur rekomendasi
1. Sinyal ini pada dasarnya **"beli alt yang sedang kuat saat pasar risk-on"** — hasilnya mengikuti arah pasar alt.
   Saat alt melemah (2022, 2025) sinyal rugi walaupun BTC naik.
2. Jangan klaim sinyal "memilih coin pemenang"; win rate ~45% dengan alpha median negatif.
3. Langkah berikut yang paling menjanjikan: **notebook 05 — exit hold tetap vs exit skor** (dipilih di dev,
   cek 2024, dengan stop keras tetap ada), dan **status kesehatan pasar alt (breadth)** sebagai info/peringatan
   tambahan di tampilan (informasi, bukan filter yang diklaim terbukti).